# 04. モデル作成: LightGBMのほかに4つのモデルを作り、予測の違いを比べる

**このノートブックの目的:** 03で作った36個の特徴量のLightGBMに、仕組みの違うモデルを足す候補を作り、単体の強さと予測の似かたを比べる。

**使うもの:** 03で保存したLightGBMのOOF(`data/oof_lightgbm_default.npy`)。CatBoost・RealMLP・TabPFNはKaggleのGPUで学習したOOF・testの予測(`data/oof_catboost_tuned.npy`・`oof_realmlp.npy`・`oof_tabpfn.npy`)。

**結論:** 5つのモデルの単体の強さはほぼ同じ(OOF AUC 0.9604〜0.9612)だった。予測の似かたは違い、木どうし(LightGBM・XGBoost・CatBoost)は似ていて、RealMLP・TabPFNは木と違う予測をした。

| モデル | 仕組み | 学習した場所 |
|---|---|---|
| LightGBM | 決定木を1本ずつ足す(葉ごとに伸ばす) | 手元のCPU |
| XGBoost | 決定木を1本ずつ足す(深さごとに伸ばす) | 手元のCPU |
| CatBoost | 決定木を1本ずつ足す(左右対称の木、カテゴリを独自の方法で数値化) | KaggleのGPU |
| RealMLP | ニューラルネット | KaggleのGPU(公開ノートブックをもとにした) |
| TabPFN | 表データ用に事前学習されたモデル(学習データを文脈として読み、学習しないで予測する) | KaggleのGPU(公開ノートブックを参考にした) |

実際の順番は、LightGBMの伸びが鈍った時点で、まずRealMLPを足し(ここでPublic LBの横ばいを抜けた)、LightGBMのハイパラ調整(05)のあとで、XGBoost・CatBoost・TabPFNを作った。

## 準備

- ライブラリと、共通処理(`src/`の自分で書いたファイル)を読み込む。`src/features.py` はデータの読み込みと特徴量の作成、`src/modeling.py` はモデルの学習と評価。
- train・testを読み込み、目的変数 `satisfaction` を0/1の配列 `y` にする。

In [1]:
import sys

sys.path.insert(0, "../src")  # 共通処理(features.py・modeling.py・public_oof.py)を読み込む
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import features
import modeling

plt.rcParams["font.family"] = "Meiryo"  # グラフの日本語ラベル用
DATA = features.DATA_DIR
train, test = features.load_train_and_test()
y = train[features.TARGET].astype(int).values
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")

train 699,635行・test 299,844行、満足率 0.444


## 1. LightGBM(03の36個、既定のハイパラ)

03で学習したOOFを読み込む。

In [2]:
predictions = {"LightGBM": np.load(DATA / "oof_lightgbm_default.npy")}
print(f"LightGBM OOF AUC {roc_auc_score(y, predictions['LightGBM']):.6f}")

LightGBM OOF AUC 0.960408


## 2. XGBoost

**目的:** LightGBMと同じ決定木の仲間だが、木の伸ばし方が違う(深さごとに伸ばす)。同じ特徴量でどれくらい違う予測になるかを見る。

**検証方法:** 同じ36個・同じ5分割で、既定に近いハイパラ(深さ6、学習率0.05、早期停止100本)で学習する。カテゴリ列はXGBoostのカテゴリ機能で扱う。

In [3]:
X, X_test = features.build_train_and_test_tables(train, test)
oof_xgboost, test_xgboost = modeling.cross_validate_xgboost(X, y, X_test=X_test)
np.save(DATA / "oof_xgboost_default.npy", oof_xgboost)
np.save(DATA / "test_xgboost_default.npy", test_xgboost)
predictions["XGBoost"] = oof_xgboost
print(f"XGBoost OOF AUC {roc_auc_score(y, oof_xgboost):.6f}")

fold0: AUC 0.960590 木600本 (61秒)


fold1: AUC 0.959324 木734本 (138秒)


fold2: AUC 0.961286 木776本 (230秒)


fold3: AUC 0.960618 木525本 (304秒)


fold4: AUC 0.960450 木552本 (381秒)


XGBoost OOF AUC 0.960447


## 3. CatBoost(KaggleのGPU)

**目的:** CatBoostは、カテゴリの特徴量を独自の方法(答えを行の順番に使って集計する)で数値化し、左右対称の木を使う。ほかの木とは違う予測になる可能性がある。

**検証方法:**
- 手元のCPUでは数時間かかるため、KaggleのGPUで学習した(`src/kaggle_gpu/train_catboost.py`)。入力は36個の特徴量を書き出したCSV(下のセル)。
- 公開ノートブックの「全特徴量をカテゴリ扱い+Target Encodingなし」(単体+0.00015の報告)を2つに分けて試し、全特徴量をカテゴリ扱いにする方だけを採用した(+0.000187)。Target Encodingを外す方は±0だった。
- ハイパラは05で調整したもの(KaggleのGPUでOptuna 30試行)。読み込むのはその設定のOOF。

In [4]:
# Kaggleに渡す入力(値ごとの満足率の2列はキーのまま。foldごとの置き換えはKaggleのスクリプトの中で行う)
pd.concat([train[["id"]], X, pd.Series(y, name=features.TARGET)], axis=1).to_csv(DATA / "features36_train.csv", index=False)
pd.concat([test[["id"]], X_test], axis=1).to_csv(DATA / "features36_test.csv", index=False)
predictions["CatBoost"] = np.load(DATA / "oof_catboost_tuned.npy")
print(f"CatBoost OOF AUC {roc_auc_score(y, predictions['CatBoost']):.6f}")

CatBoost OOF AUC 0.960936


## 4. RealMLP(ニューラルネット、KaggleのGPU)

**目的:** 03の最後で、特徴量を足してもOOFの伸びが鈍り、Public LBは4日間横ばいになった。決定木とは仕組みの違うニューラルネットなら、木が苦手な部分を補えると考えた。

**検証方法:** yekenot氏の公開ノートブック『PS|S6|E10: RealMLP · PyTabKit』(pytabkitのRealMLP_TD_Classifier)をもとに、次の4つを変えてKaggleのGPUで学習した。公開ノートブックの再現なので、コードはこのリポジトリに含めない。
1. 最良のエポック(学習の回数)を、採点用のfoldではなく学習側の5%で選ぶ(`val_fraction=0.05`)。採点用で選ぶとOOFが甘くなるため。
2. 5分割の分け方をLightGBMと同じにする(`StratifiedKFold(5, shuffle=True, random_state=42)`)。同じ行どうしで比べ、混ぜられるようにするため。
3. エポック数を3→4にする(`n_epochs=4`)。
4. モデルのseed 42と1の2本を作り、予測を平均する。

In [5]:
predictions["RealMLP"] = np.load(DATA / "oof_realmlp.npy")
print(f"RealMLP OOF AUC {roc_auc_score(y, predictions['RealMLP']):.6f}")

RealMLP OOF AUC 0.961192


## 5. TabPFN(KaggleのGPU)

**目的:** 木ともニューラルネットとも仕組みの違うモデルを足す。TabPFNは表データ用に事前学習されたモデル(TabPFN-3.5、`tabpfn==9.0.0`)で、学習foldの行を「文脈」として読み、検証の行を予測する。

**検証方法:** hemingweb氏の公開ノートブック『S6E10 EXP01 TabPFN35 Full Context』を参考にしたため、コードは含めない。入力は元の特徴量21個(カテゴリ4個はカテゴリとして指定)、文脈は学習foldの全行、`n_estimators=1`・`random_state=0`。5分割はLightGBMと同じ。KaggleのGPU(T4)で1foldあたり約1.5時間かかった。

In [6]:
predictions["TabPFN"] = np.load(DATA / "oof_tabpfn.npy")
print(f"TabPFN OOF AUC {roc_auc_score(y, predictions['TabPFN']):.6f}")

TabPFN OOF AUC 0.960844


## 6. 単体の強さと予測の似かた

**目的:** 混ぜて伸びるのは「強さが近く、予測が違う」モデルどうし。単体のOOF AUCと、予測の順位相関(1に近いほど似た予測)を並べて見る。

In [7]:
single = pd.Series({name: roc_auc_score(y, p) for name, p in predictions.items()}, name="単体のOOF AUC")
correlation = modeling.rank_correlation_table(predictions)
display(single.round(6).to_frame())
correlation.round(3)

,単体のOOF AUC
LightGBM,0.960408
XGBoost,0.960447
CatBoost,0.960936
RealMLP,0.961192
TabPFN,0.960844


,LightGBM,XGBoost,CatBoost,RealMLP,TabPFN
LightGBM,1.000,0.985,0.979,0.962,0.959
XGBoost,0.985,1.000,0.980,0.963,0.959
CatBoost,0.979,0.980,1.000,0.971,0.963
RealMLP,0.962,0.963,0.971,1.000,0.966
TabPFN,0.959,0.959,0.963,0.966,1.000


#### 結果と考察

- 単体の強さはほぼ同じで、RealMLPが最も強い。
- 木どうし(LightGBM・XGBoost・CatBoost)の順位相関は高く、似た予測をしている。RealMLP・TabPFNは木との相関が低く、木と違う予測をしている。混ぜたときに補い合う候補になる(06で確かめる)。

## Summary

| モデル | 役割 |
|---|---|
| LightGBM・XGBoost・CatBoost | 同じ36個の特徴量の決定木。互いに似た予測 |
| RealMLP | ニューラルネット。単体で最も強く、木と違う予測 |
| TabPFN | 事前学習済みの表データ用モデル。元の特徴量21個だけで、木と違う予測 |

次は05で、LightGBM・XGBoost・CatBoostのハイパラを調整する。